In [ ]:
%pip install protobuf==3.20.3

In [ ]:
%pip install torch \
             torchaudio \
             transformers \
             pandas \
             whisper_timestamped \
             gdown \
             Levenshtein \
             joblib \
             scikit-learn \
             soundfile

In [ ]:
%pip install "numpy<2.0"

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import torchaudio
from transformers import WavLMModel


DATA_PATH = "/kaggle/input/switchboard-preprocessed-dataset-5gb/PreProcessedData5GB" 
BATCH_SIZE = 4      
LR = 1e-5           
EPOCHS = 10         
MAX_AUDIO_LEN = 5   
SR = 16000          

CLASS_NAMES = [
    "Filled Pause (FP)", # Class 0
    "Phrase Rep (RP)",   # Class 1
    "Revision (RV)",     # Class 2
    "Restart (RS)",      # Class 3
    "Partial Word (PW)"  # Class 4
]

class DisfluencyDataset(Dataset):
    def __init__(self, file_list, base_path, max_seconds=5):
        self.file_list = file_list
        self.base_path = base_path
        self.max_frames_8k = int(max_seconds * 8000) 
        self.target_label_frames = int(max_seconds * 50) 

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]
        audio_path = os.path.join(self.base_path, "wav_sil", filename)
        
        # 1. Load Audio (It is 8kHz, so it loads fast)
        waveform, sample_rate = torchaudio.load(audio_path)
                
        # 2. Pad/Truncate to 8k length (40,000 samples)
        if waveform.shape[1] > self.max_frames_8k:
            waveform = waveform[:, :self.max_frames_8k]
        else:
            pad_len = self.max_frames_8k - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        # 3. Load Labels (Unchanged)
        label_name = filename.replace('.wav', '.npy')
        label_path = os.path.join(self.base_path, "labels_framelevel", label_name)
        full_labels = np.load(label_path) 
        
        # Select correct columns [FP, RP, RV, RS, PW]
        target_indices = [3, 4, 5, 6, 7] 
        labels = full_labels[:, target_indices]
        labels = torch.FloatTensor(labels)

        # Align Labels
        current_frames = labels.shape[0]
        if current_frames > self.target_label_frames:
            labels = labels[:self.target_label_frames, :]
        else:
            pad_rows = self.target_label_frames - current_frames
            padding = torch.zeros((pad_rows, 5)) 
            labels = torch.cat((labels, padding), dim=0)

        return waveform.squeeze(0), labels


class AcousticModel(nn.Module):
    def __init__(self):
        super(AcousticModel, self).__init__()
        self.basemodel = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
        self.linear = nn.Linear(768, 5) # 5 Classes

    def forward(self, x):
        # Ensure input is [Batch, Time] (remove extra 1s if present)
        if x.dim() > 2: 
            x = x.reshape(x.shape[0], -1)
            
        feats = self.basemodel.feature_extractor(x)
        feats = feats.transpose(1, 2)
        feats, _ = self.basemodel.feature_projection(feats)
        emb = self.basemodel.encoder(feats, return_dict=True)[0]
        out = self.linear(emb)
        return emb, out

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # --- A. Data Setup ---
    wav_dir = os.path.join(DATA_PATH, "wav_sil")
    all_files = [f for f in os.listdir(wav_dir) if f.endswith('.wav')]
    
    # Split
    train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)
    print(f"Training on {len(train_files)} files | Testing on {len(test_files)} files")

    train_ds = DisfluencyDataset(train_files, DATA_PATH, max_seconds=MAX_AUDIO_LEN)
    test_ds = DisfluencyDataset(test_files, DATA_PATH, max_seconds=MAX_AUDIO_LEN)

    # Kaggle usually gives 2-4 CPU cores. Set workers to 2 or 4.
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    

    

    # --- B. Model Setup ---
    model = AcousticModel()
    model.to(device)

    gpu_resampler = torchaudio.transforms.Resample(orig_freq=8000, new_freq=16000).to(device)

    # Freeze CNN layers
    for param in model.basemodel.feature_extractor.parameters():
        param.requires_grad = False

    # --- C. Training Loop ---
    criterion = nn.BCEWithLogitsLoss() 
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

    print("\nStarting Training with GPU Resampling...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        
        for batch_idx, (audio, targets) in enumerate(train_loader):
            # 1. Move raw 8k audio to GPU
            audio = audio.to(device) 
            targets = targets.to(device)
            
            # 2. RESAMPLE ON GPU 
            # Input: [Batch, 40000] -> Output: [Batch, 80000]
            audio_16k = gpu_resampler(audio)
            
            # 3. Feed 16k audio to Model
            _, outputs = model(audio_16k)
            
            min_len = min(outputs.shape[1], targets.shape[1])
            outputs = outputs[:, :min_len, :]
            targets = targets[:, :min_len, :]

            loss = criterion(outputs, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "wavlm_5gb_finetuned.pt")
    print("\nModel Saved!")

    print("\nGenerating Metrics on Test Set...")
    model.eval()
    
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for audio, targets in test_loader:
            audio, targets = audio.to(device), targets.to(device)
            
            _, outputs = model(audio)
            
            # Convert logits to class indices (argmax)
            # Shape: [Batch, Time, 5] -> [Batch, Time]
            preds = torch.argmax(outputs, dim=2)
            true_labels = torch.argmax(targets, dim=2)
            
            # Flatten to list of frames for metrics
            # We only take the valid length where shapes match
            min_len = min(preds.shape[1], true_labels.shape[1])
            
            # Add to lists (CPU numpy)
            y_pred_all.extend(preds[:, :min_len].reshape(-1).cpu().numpy())
            y_true_all.extend(true_labels[:, :min_len].reshape(-1).cpu().numpy())

    # 1. Classification Report
    print("\n--- 1. Classification Report ---")
    print(classification_report(y_true_all, y_pred_all, target_names=CLASS_NAMES, digits=4))

    # 2. F1 Score (Macro & Weighted)
    f1_macro = f1_score(y_true_all, y_pred_all, average='macro')
    print(f"2. F1 Score (Macro): {f1_macro:.4f}")

    # 3. Precision (Macro)
    prec_macro = precision_score(y_true_all, y_pred_all, average='macro')
    print(f"3. Precision (Macro): {prec_macro:.4f}")

    # 4. Recall (Macro)
    rec_macro = recall_score(y_true_all, y_pred_all, average='macro')
    print(f"4. Recall (Macro): {rec_macro:.4f}")

    # 5. UAR (Unweighted Average Recall)
    print(f"5. UAR (Unweighted Average Recall): {rec_macro:.4f}")

    # 6. Confusion Matrix
    print("\n--- 6. Confusion Matrix ---")
    cm = confusion_matrix(y_true_all, y_pred_all)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import numpy as np
# import torchaudio
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, recall_score
# from transformers import WavLMModel

# DATA_PATH = "/kaggle/input/switchboard-preprocessed-dataset-5gb/PreProcessedData5GB" 
# BATCH_SIZE = 4      
# MAX_AUDIO_LEN = 5   
# SR = 16000          
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CLASS_NAMES = ["Filled Pause (FP)", "Phrase Rep (RP)", "Revision (RV)", "Restart (RS)", "Partial Word (PW)"]

# # 2. DATASET CLASS
# class DisfluencyDataset(Dataset):
#     def __init__(self, file_list, base_path, max_seconds=5):
#         self.file_list = file_list
#         self.base_path = base_path
#         self.max_frames_8k = int(max_seconds * 8000) 
#         self.target_label_frames = int(max_seconds * 50) 

#     def __len__(self):
#         return len(self.file_list)

#     def __getitem__(self, idx):
#         filename = self.file_list[idx]
#         audio_path = os.path.join(self.base_path, "wav_sil", filename)
        
#         # Load Audio (8k)
#         waveform, sample_rate = torchaudio.load(audio_path)
        
#         # Pad/Truncate
#         if waveform.shape[1] > self.max_frames_8k:
#             waveform = waveform[:, :self.max_frames_8k]
#         else:
#             pad_len = self.max_frames_8k - waveform.shape[1]
#             waveform = torch.nn.functional.pad(waveform, (0, pad_len))

#         # Load Labels
#         label_name = filename.replace('.wav', '.npy')
#         label_path = os.path.join(self.base_path, "labels_framelevel", label_name)
#         full_labels = np.load(label_path) 
        
#         # Select columns [FP, RP, RV, RS, PW]
#         target_indices = [3, 4, 5, 6, 7] 
#         labels = full_labels[:, target_indices]
#         labels = torch.FloatTensor(labels)

#         # Align Labels
#         current_frames = labels.shape[0]
#         if current_frames > self.target_label_frames:
#             labels = labels[:self.target_label_frames, :]
#         else:
#             pad_rows = self.target_label_frames - current_frames
#             padding = torch.zeros((pad_rows, 5)) 
#             labels = torch.cat((labels, padding), dim=0)

#         return waveform.squeeze(0), labels

# # 3. MODEL ARCHITECTURE
# class AcousticModel(nn.Module):
#     def __init__(self):
#         super(AcousticModel, self).__init__()
#         self.basemodel = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
#         self.linear = nn.Linear(768, 5) 

#     def forward(self, x):
#         if x.dim() > 2: x = x.reshape(x.shape[0], -1)
#         feats = self.basemodel.feature_extractor(x)
#         feats = feats.transpose(1, 2)
#         feats, _ = self.basemodel.feature_projection(feats)
#         emb = self.basemodel.encoder(feats, return_dict=True)[0]
#         out = self.linear(emb)
#         return emb, out

# # 4. LOAD DATA & MODEL
# print("Initializing Data Loader...")
# wav_dir = os.path.join(DATA_PATH, "wav_sil")
# all_files = [f for f in os.listdir(wav_dir) if f.endswith('.wav')]
# # Use same seed as training to ensure we get the correct Test Set
# _, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

# test_ds = DisfluencyDataset(test_files, DATA_PATH, max_seconds=MAX_AUDIO_LEN)
# # num_workers=2 is safe for Kaggle
# test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# print(" Loading Model...")
# model = AcousticModel()
# model.to(DEVICE)

# # Load Weights
# try:
#     model.load_state_dict(torch.load("wavlm_5gb_finetuned.pt", map_location=DEVICE))
#     print(" Successfully loaded saved weights!")
# except FileNotFoundError:
#     print(" Model file not found. Ensure 'wavlm_5gb_finetuned.pt' is in /kaggle/working/")

# # 5. RUN EVALUATION
# print("\n Running Evaluation on Test Set...")
# model.eval()
# gpu_resampler = torchaudio.transforms.Resample(orig_freq=8000, new_freq=16000).to(DEVICE)

# y_true_all = []
# y_pred_all = []

# with torch.no_grad():
#     for audio, targets in test_loader:
#         audio = audio.to(DEVICE)
#         targets = targets.to(DEVICE)
        
#         # Resample on GPU
#         audio_16k = gpu_resampler(audio)
        
#         _, outputs = model(audio_16k)
        
#         # Align shapes
#         min_len = min(outputs.shape[1], targets.shape[1])
#         outputs = outputs[:, :min_len, :]
#         targets = targets[:, :min_len, :]
        
#         # Sigmoid Thresholding
#         probs = torch.sigmoid(outputs)
#         preds = (probs > 0.5).float()
        
#         y_pred_all.append(preds.reshape(-1, 5).cpu().numpy())
#         y_true_all.append(targets.reshape(-1, 5).cpu().numpy())

# # Concatenate
# y_pred_all = np.vstack(y_pred_all)
# y_true_all = np.vstack(y_true_all)

# # 6. REPORT
# print("\n" + "="*60)
# print("FINAL EVALUATION REPORT")
# print("="*60)
# print(classification_report(y_true_all, y_pred_all, target_names=CLASS_NAMES, zero_division=0))

# recalls = recall_score(y_true_all, y_pred_all, average=None, zero_division=0)
# uar = recalls.mean()

# print("-" * 60)
# print(f"Recalls per class: {recalls}")
# print(f"Unweighted Average Recall (UAR): {uar:.4f}")
# print("-" * 60)